# Notebook for measuring runtime of Hashing, bucketing and similarity value computation 

In [35]:
import os
import sys
import numpy as np
import itertools
import pandas as pd

def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Example usage
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root found: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from utils.helpers.measure_similarities import *


Project root found: /cluster/home/eivindoy/masteroppgave


# Code for running several combinations of the runtime parameters

# Disk

In [36]:
MEASURE="disk_dtw_cy"
CITY="porto"
DATA_SIZE = [1000]

#Strategies
BUCKETING_METHOD = "loose"
TRUE_TRAJECTORIES = True

#Logistics
PARALLEL_JOBS = 24
ITERATIONS = 1

In [12]:
# bulk_groups = [
#     {
#         "name": "DIA_1-1.4",
#         "DIAMETER_VALUES": [0.6, 0.8, 1, 1.2, 1.4],
#         "LAYERS_VALUES": [1, 2, 3, 4, 5, 6],
#         "DISKS_VALUES": [10, 20, 30, 40, 50]
#     }
# ]


In [37]:
if "disk_dtw_cy" or "disk_frechet_cy" in MEASURE:
    SCHEME = "disk"
elif "grid_dtw_cy" or "grid_frechet_cy" in MEASURE:
    SCHEME = "grid"
    
if "dtw" in MEASURE:
    measure = "dtw"
elif "frechet" in MEASURE:
    measure = "frechet"

#Filenames
if TRUE_TRAJECTORIES:
    folder = "true_trajectories"
    file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(true_trajectories)_{CITY}_{measure}_{DATA_SIZE}_{SCHEME}.csv"
else:
    folder = "hashed_trajectories"
    file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(hashed_trajectories)_{CITY}_{measure}_{DATA_SIZE}_{SCHEME}.csv"

output_path = f"../../../results_hashed/runtimes/bucketing/{CITY}/{folder}/{BUCKETING_METHOD}/{measure}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)


# DATA_SIZE_FILE_NAME = DATA_SIZE[0]  # Use only the first size, as before

DATA_SIZE = DATA_SIZE[0]  # Use only the first size, as before

bucket_evaluation_file = bucket_evaluation_file = f"../../../results_hashed/bucket_evaluation/{BUCKETING_METHOD}/{CITY}_{measure}_{SCHEME}_{DATA_SIZE}_{BUCKETING_METHOD}.csv"

In [ ]:
import pandas as pd

# Load bucket evaluation data
bucket_evaluation_df = pd.read_csv(bucket_evaluation_file)

# Filter rows where Recall == 1 and Threshold == 0.1
filtered_df = bucket_evaluation_df[
    (bucket_evaluation_df["Avg Recall"] == 1) &
    (bucket_evaluation_df["Threshold"] == 0.1)
]

data_size = DATA_SIZE

first_write = True  # Write header only once

print(f" Current param config: \n \tBUCKETING: YES \n\tBUCKETING_METHOD: {BUCKETING_METHOD}\n\tTRUE_TRAJECTORIES: {TRUE_TRAJECTORIES}\n\tCITY: {CITY}\n\tMEASURE: {MEASURE}\n\tDATA_SIZE: {DATA_SIZE}\n\tSCHEME: {SCHEME} \n\tPARALLEL_JOBS: {PARALLEL_JOBS} \n\tITERATIONS: {ITERATIONS} \n\n")

for _, row in filtered_df.iterrows():
    diameter = row["Diameter"]
    layers = row["Layers"]
    disks = row["Disks"]

    print(f" \n Running for Diameter: {diameter}, Layers: {layers}, Disks: {disks}, Data Size: {data_size}, True Trajectories: {TRUE_TRAJECTORIES}")

    # Run correct function based on whether we're using true trajectories
    if TRUE_TRAJECTORIES:
        df_result = compute_hashed_similarity_runtimes_with_bucketing_with_true_sim(
            measure=MEASURE,
            city=CITY,
            diameter=diameter,
            layers=layers,
            disks=disks,
            parallel_jobs=PARALLEL_JOBS,
            data_size=data_size,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )
    else:
        df_result = compute_hashed_similarity_runtimes_with_bucketing(
            measure=MEASURE,
            city=CITY,
            diameter=diameter,
            layers=layers,
            disks=disks,
            parallel_jobs=PARALLEL_JOBS,
            data_size=data_size,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )

    # Add parameters to result DataFrame
    df_result["City"] = CITY
    df_result["Measure"] = measure
    df_result["Diameter"] = diameter
    df_result["Layers"] = layers
    df_result["Disks"] = disks
    df_result["Size"] = data_size

    # Reorder columns
    desired_order = ["City", "Measure", "Diameter", "Layers", "Disks", "Size",
                     "Average Similarity Computation Time (Seconds)",
                     "Average Hash Generation Time (Seconds)",
                     "Average Bucket Distribution Time (Seconds)",
                     "Total time (Seconds)"]
    df_result = df_result[desired_order]

    # Append to CSV
    df_result.to_csv(output_path, mode='a', header=first_write, index=False)
    first_write = False



 Current param config: 
 	BUCKETING: YES 
	BUCKETING_METHOD: loose
	TRUE_TRAJECTORIES: True
	CITY: porto
	MEASURE: disk_dtw_cy
	DATA_SIZE: 1000
	SCHEME: disk 
	PARALLEL_JOBS: 24 
	ITERATIONS: 1 


 
 Running for Diameter: 0.6, Layers: 6, Disks: 50, Data Size: 1000, True Trajectories: True
Computing disk_dtw_cy for porto with 24 jobs - Iteration 1/1
 
 Running for Diameter: 0.8, Layers: 3, Disks: 50, Data Size: 1000, True Trajectories: True
Computing disk_dtw_cy for porto with 24 jobs - Iteration 1/1
 
 Running for Diameter: 0.8, Layers: 4, Disks: 50, Data Size: 1000, True Trajectories: True
Computing disk_dtw_cy for porto with 24 jobs - Iteration 1/1
 
 Running for Diameter: 0.8, Layers: 5, Disks: 50, Data Size: 1000, True Trajectories: True
Computing disk_dtw_cy for porto with 24 jobs - Iteration 1/1
 
 Running for Diameter: 0.8, Layers: 6, Disks: 40, Data Size: 1000, True Trajectories: True
Computing disk_dtw_cy for porto with 24 jobs - Iteration 1/1
 
 Running for Diameter: 0.8, Lay

# Grid

In [ ]:
MEASURE = "grid_dtw_cy"
CITY = "porto"
DATA_SIZE = [2500]

# Strategies
BUCKETING_METHOD = "loose"
TRUE_TRAJECTORIES = True

# Logistics
PARALLEL_JOBS = 24
ITERATIONS = 1

In [ ]:
# # bulk_groups = [
# #     {
# #         "name": "RES_0.01-0.15",
# #         "RESOLUTION_VALUES": [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15],
# #         "LAYERS_VALUES": [1, 2, 3, 4]
# #     },
# #     {
# #         "name": "EDGE",
# #         "RESOLUTION_VALUES": [0.5, 1],
# #         "LAYERS_VALUES": [1, 2, 3, 4, 5, 6, 10]
# #     }
# # ]

# bulk_groups = [
#     {
#         "name": "1",
#         "RESOLUTION_VALUES": [0.04],
#         "LAYERS_VALUES": [3]
#     },
#     {
#         "name": "2",
#         "RESOLUTION_VALUES": [0.05],
#         "LAYERS_VALUES": [2]
#     },
#     {
#         "name": "3",
#         "RESOLUTION_VALUES": [0.06],
#         "LAYERS_VALUES": [2]
#     },
#     {
#         "name": "4",
#         "RESOLUTION_VALUES": [0.07],
#         "LAYERS_VALUES": [2]
#     },
#     {
#         "name": "5",
#         "RESOLUTION_VALUES": [0.08],
#         "LAYERS_VALUES": [1]
#     },
#     {
#         "name": "6",
#         "RESOLUTION_VALUES": [0.09],
#         "LAYERS_VALUES": [1]
#     }
# ]

In [ ]:
# Determine SCHEME and measure type
if "disk_dtw_cy" in MEASURE or "disk_frechet_cy" in MEASURE:
    SCHEME = "disk"
elif "grid_dtw_cy" in MEASURE or "grid_frechet_cy" in MEASURE:
    SCHEME = "grid"

if "dtw" in MEASURE:
    measure = "dtw"
elif "frechet" in MEASURE:
    measure = "frechet"

# File path setup
if TRUE_TRAJECTORIES:
    folder = "true_trajectories"
    file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(true_trajectories)_{CITY}_{measure}_{DATA_SIZE}_{SCHEME}.csv"
else:
    folder = "hashed_trajectories"
    file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(hashed_trajectories)_{CITY}_{measure}_{DATA_SIZE}_{SCHEME}.csv"

output_path = f"../../../results_hashed/runtimes/bucketing/{CITY}/{folder}/{BUCKETING_METHOD}/{measure}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

DATA_SIZE = DATA_SIZE[0]  # Use only the first size, as before

bucket_evaluation_file = bucket_evaluation_file = f"../../../results_hashed/bucket_evaluation/{BUCKETING_METHOD}/{CITY}_{measure}_{SCHEME}_{DATA_SIZE}_{BUCKETING_METHOD}.csv"

In [ ]:
import pandas as pd

# Load bucket evaluation data for the grid scheme
bucket_evaluation_df = pd.read_csv(bucket_evaluation_file)

# Filter for perfect recall and threshold 0.1
filtered_df = bucket_evaluation_df[
    (bucket_evaluation_df["Avg Recall"] == 1) &
    (bucket_evaluation_df["Threshold"] == 0.1)
]

data_size = DATA_SIZE
first_write = True

print(f" Current param config: \n \tBUCKETING: YES \n\tBUCKETING_METHOD: {BUCKETING_METHOD}\n\tTRUE_TRAJECTORIES: {TRUE_TRAJECTORIES}\n\tCITY: {CITY}\n\tMEASURE: {MEASURE}\n\tDATA_SIZE: {DATA_SIZE}\n\tSCHEME: {SCHEME} \n\tPARALLEL_JOBS: {PARALLEL_JOBS} \n\tITERATIONS: {ITERATIONS} \n\n")

for _, row in filtered_df.iterrows():
    resolution = row["Resolution"]
    layers = row["Layers"]

    print(f"\nRunning for Resolution: {resolution}, Layers: {layers}, Data Size: {data_size}, True Trajectories: {TRUE_TRAJECTORIES}")

    if TRUE_TRAJECTORIES:
        df_result = compute_hashed_similarity_runtimes_with_bucketing_with_true_sim(
            measure=MEASURE,
            city=CITY,
            res=resolution,
            layers=layers,
            parallel_jobs=PARALLEL_JOBS,
            data_size=data_size,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )
    else:
        df_result = compute_hashed_similarity_runtimes_with_bucketing(
            measure=MEASURE,
            city=CITY,
            res=resolution,
            layers=layers,
            parallel_jobs=PARALLEL_JOBS,
            data_size=data_size,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )

    # Add parameters
    df_result["City"] = CITY
    df_result["Measure"] = MEASURE
    df_result["Resolution"] = resolution
    df_result["Layers"] = layers
    df_result["Size"] = data_size

    # Reorder columns
    desired_order = [
        "City", "Measure", "Resolution", "Layers", "Size",
        "Average Similarity Computation Time (Seconds)",
        "Average Hash Generation Time (Seconds)",
        "Average Bucket Distribution Time (Seconds)",
        "Total time (Seconds)"
    ]
    df_result = df_result[desired_order]

    # Write to file
    df_result.to_csv(output_path, mode='a', header=first_write, index=False)
    first_write = False


 Current param config: 
 	BUCKETING: YES 
	BUCKETING_METHOD: loose
	TRUE_TRAJECTORIES: True
	CITY: porto
	MEASURE: grid_dtw_cy
	DATA_SIZE: [2500]
	SCHEME: grid 
	PARALLEL_JOBS: 24 
	ITERATIONS: 1 



Running for Resolution: 0.04, Layers: 3, Data Size: 2500, True Trajectories: True
Computing grid_dtw_cy for porto with 24 jobs - Iteration 1/1

Running for Resolution: 0.05, Layers: 2, Data Size: 2500, True Trajectories: True
Computing grid_dtw_cy for porto with 24 jobs - Iteration 1/1
